In [ ]:
!pip install -q openai tqdm
import json, re, time, random, os
from collections import defaultdict, Counter
from google.colab import userdata
from openai import OpenAI
from tqdm import tqdm

client = OpenAI(api_key=userdata.get('WORKIT_API'))
MODEL = "gpt-4o"

N    = 3       # 시드 1건당 변형 수
TEMP = 0.9     # 다양성 우선, few-shot + _audit 검증으로 판정보존 리스크 상쇄


In [ ]:
# 시드 로드 + train/valid 분할
raw = json.load(open('rpt_seed_golden.json', encoding='utf-8'))
seed_all = raw['data'] if isinstance(raw, dict) and 'data' in raw else raw

by_label = defaultdict(list)
for r in seed_all:
    by_label[r['output']['label']].append(r)

random.seed(42)
seed_train, seed_valid = [], []
VALID_RATIO = 0.30
for label, items in by_label.items():
    random.shuffle(items)
    n_valid = max(1, round(len(items) * VALID_RATIO))
    seed_valid += items[:n_valid]
    seed_train += items[n_valid:]

random.shuffle(seed_train); random.shuffle(seed_valid)
print(f"train {len(seed_train)} / valid {len(seed_valid)}")
print("train:", dict(Counter(r['output']['label'] for r in seed_train)))
print("valid:", dict(Counter(r['output']['label'] for r in seed_valid)))

json.dump(seed_valid, open('RPT_valid.json','w'), ensure_ascii=False, indent=2)
print("저장: RPT_valid.json (순수, 증강 안 함)")

In [ ]:
# valid.json 있는 경우

# raw = json.load(open('rpt_seed_golden.json', encoding='utf-8'))
# seed_all = raw['data'] if isinstance(raw, dict) and 'data' in raw else raw

# # 원본 시드 자체의 seed_id 중복 여부 사전 검증 (기존과 동일)
# seed_id_counts = Counter(r.get('seed_id', r.get('id')) for r in seed_all)
# dup_seed_ids = [k for k, v in seed_id_counts.items() if v > 1]
# if dup_seed_ids:
#     print(f"⚠ 원본 시드에 중복 seed_id {len(dup_seed_ids)}건 발견 → 인덱스 부여로 강제 유일화")
#     print(dup_seed_ids[:10])
#     for idx, r in enumerate(seed_all):
#         base = r.get('seed_id', r.get('id'))
#         r['seed_id'] = f"{base}__{idx}"
# else:
#     for r in seed_all:
#         if 'seed_id' not in r:
#             r['seed_id'] = r['id']

# # ★변경: 랜덤 재분할 대신, 업로드한 RPT_valid.json에 실제로 있는 seed_id를 기준으로 제외
# VALID_PATH = '/content/rpt_vaild.json'  # 업로드한 실제 경로로 바꾸세요
# valid_raw = json.load(open(VALID_PATH, encoding='utf-8'))
# seed_valid = valid_raw['data'] if isinstance(valid_raw, dict) and 'data' in valid_raw else valid_raw

# valid_ids = {r.get('seed_id', r.get('id')) for r in seed_valid}
# seed_train = [r for r in seed_all if r.get('seed_id', r.get('id')) not in valid_ids]

# print(f"train {len(seed_train)} / valid {len(seed_valid)}")
# print("train:", dict(Counter(r['output']['label'] for r in seed_train)))
# print("valid:", dict(Counter(r['output']['label'] for r in seed_valid)))

In [ ]:
# ============================================================
# 증강 프롬프트
# ============================================================
RPT_AUG_PROMPT = """당신은 공공 SW '사업추진결과보고서(RPT)' 검토 학습데이터를 증강하는 생성기다.
골든 시드 1건을 입력받아, 원본 판정을 유지하는 변형 케이스 N건을 JSON 배열로만 출력한다.

[대전제]
- pep_excerpt(계획)와 rpt_excerpt(결과보고)를 대조해 criteria별로 충족/불가/검토를 판정한다.
- refs, 법조문, disclaimer, note 같은 필드는 만들지 마라.
- input.criteria는 원본과 100% 동일하게 유지한다.
- output.label과 output.eval의 criteria별 판정은 원본과 100% 동일하게 유지한다.
- 전체 label 계산: 불가 1개 이상 → 불가 / 불가 없고 검토 1개 이상 → 검토 / 전부 충족 → 충족

[eval 형식 규칙]
- output.eval은 문자열 배열, 각 원소는 "특성: 판정 — 사유" 형식.
- 특성명은 input.criteria와 철자까지 동일해야 한다.
- 정확성(ACC)에는 "검토"가 존재하지 않는다. 반드시 충족 또는 불가 둘 중 하나다.

[criteria별 판정 기준]

1. 완전성(COMP)
- 충족: PEP 과업·범위 항목이 RPT에 모두 대응(동의어·패러프레이즈 인정)
- 불가: ①PEP 항목 중 하나 이상이 RPT 어디에도 없음(유사성조차 없음) ②조건부 문구(재량 여부 불문 불가)
- 검토: 표현은 다른데 동일 항목인지 판단이 갈리는 경우만
- 판단절차: 같은 산출물·주체·시점이면 충족 / 완전히 다르면 불가 / 애매하면 검토

2. 정확성(ACC) — 검토 없음, 충족/불가 이분
- 충족: 수치·규격이 PEP·계획과 일치
- 불가 유형(반드시 아래 3가지 중 정확히 하나):
  유형1) PEP 기준값과 RPT 결과값이 다름(즉시 불가)
  유형2) RPT 문서 내부 자기모순(실측치가 목표 미달인데 "충족"이라 서술)
  유형3) PEP·RPT 어디에도 비교할 기준값 자체가 없음

3. 검증가능성(VER)
- 충족: 화이트리스트(수치·건수·기준일·비율·시험결과)로 성과 서술, 블랙리스트 표현 없음
- 불가: 검증 서술 자체가 없거나 항목이 통째로 비어있음
- 검토: 블랙리스트(적절히/최선을 다해/원활히 협의하여/필요한 범위에서/우수한 수준으로/성실히/무리 없이/신속히)가 하나라도 있으면 무조건 검토
  → 블랙리스트를 제거해도 유추 가능하다는 이유로 충족으로 격상시키지 마라. 유추 가능 여부는 판정에 영향 없다.

4. 추적성(TRACE)
- 충족: PEP 단계·산출물과 RPT 결과가 명확한 표현으로 1:1 확인됨
- 불가: RPT 산출물명이 PEP 어느 단계에도 대응 안 됨(또는 반대), 명칭이 완전히 달라 대응 자체가 안 보임
- 검토: 교차 대조 자료가 입력에 없음 / 명칭이 다른데 재구성됐지만 대응 가능성이 남아있어 판단이 갈리는 경우
- 판단절차: 교차대조 자료 유무 확인 → 없으면 검토 / 있으면 대응불가(불가)·유추가능(검토)·명확대응(충족) 3분기

[정확성 사유 작성 제약 — 재구성 금지]
- 정확성이 불가인 경우, 원본 사유에서 어떤 사실(PEP 목표값, RPT 실적값, 또는 "기준값 자체가 없다"는 사실)을
  근거로 들었는지 먼저 확인하라.
- 그 근거가 된 사실 관계(구체적으로 어떤 수치가 있었는지/없었는지)는 문장 표현만 바꾸고
  내용은 절대 재구성하지 마라. "PEP 목표 X, RPT 실적 Y로 미달"이라는 사실관계를
  다른 논리(예: "기준값 자체가 없다")로 바꿔치기하는 것은 paraphrase가 아니라 사실 왜곡이다.
- 허용되는 변경: 문장 순서, 어휘 선택, 문체
- 금지되는 변경: 어떤 수치가 존재했는지/없었는지에 대한 사실관계 자체

[정확성 판정 예시 — 반드시 이 패턴을 따를 것]
원본: "정확성: 불가 — PEP 목표 처리량 500건/일 대비 RPT 실적 320건/일로 목표 미달"
  (유형1: PEP값 500 vs RPT값 320, 수치가 둘 다 존재하고 서로 다름)
올바른 변형 (유형1 유지): "정확성: 불가 — 계획된 일일 처리량 500건 대비 실제 처리 실적 320건으로 기준에 미달하였다"
잘못된 변형 (유형3으로 둔갑, 절대 금지): "정확성: 불가 — 처리량에 대한 비교 기준값이 제시되지 않아 검증할 수 없다"
  → 원본엔 500/320이라는 수치가 둘 다 있었는데, 변형에서 이걸 지우고 "기준값이 없다"고 바꾸면 안 된다.

[검증가능성 판정 예시 — 반드시 이 패턴을 따를 것]
원본: "검증가능성: 검토 — 인력을 적절히 배치하였다는 서술 외 구체적 투입 인원수는 제시되지 않음"
  (블랙리스트 "적절히"가 있으므로 검토)
올바른 변형: "검증가능성: 검토 — 필요한 인력을 원활히 협의하여 배치하였다고만 서술되어 구체적 인원 규모는 확인되지 않는다"
잘못된 변형 (충족으로 격상, 절대 금지): "검증가능성: 충족 — 실무 부서와 협의하여 인력을 배치하였고 이는 통상적인 업무 절차로 확인 가능하다"
  → 블랙리스트 표현이 있으면 "그래도 유추는 되잖아"라며 충족으로 올리면 안 된다. 무조건 검토.

[사실 삽입 금지]
- eval 사유의 모든 수치·기간·건수·일자·비율은 반드시 그 변형본의 pep_excerpt/rpt_excerpt 안에 그대로 존재해야 한다.
- 발췌문에 없는 수치·기간(예: "1일 내", "3개")이나 숫자쌍(예: "170/1500")을 새로 지어내지 마라.
- 원본 사유에 없던 조건(기한, 횟수, 절차)을 임의로 추가하지 마라.

[완전성 불가 보존]
- 원본에서 완전성이 불가였던 PEP 누락 항목(특정 모듈, 데이터이관 범위 등)은 변형본의 rpt_excerpt에서도
  이름을 명시해 계속 누락 상태로 남겨야 한다. 노이즈 추가나 표현 변경으로 우연히 메워진 것처럼 보이면 안 된다.

[허용 변형 방식 — 골라 조합]
A. 표현 변형: 같은 사실·수치·판정 유지, 문장구조·어휘·순서만 변경. 표↔줄글 전환 가능.
B. 시나리오 교체: 교육청/대학/공공기관 SW 사업 맥락에서 도메인만 교체(학사행정, 학습관리, 상담챗봇, 정보보안 등). 판정 구조는 유지.
C. 무해한 노이즈 추가: 운영관리/교육훈련/개인정보보호/보안점검/인수인계/유지관리 등 정상 항목을 1~3개 추가 가능하나, 누락·불일치·모호성·추적단절을 해소하면 안 됨. 노이즈 항목의 수치는 eval 사유에서 인용 금지.

[문체 규칙]
- 교육청/대학/공공기관 SW 사업 문맥, 현실적 수치, 과거형 성과 서술("구축하였다", "수행하였다").
- 규범적 표현이나 법률 판단어는 쓰지 않는다.

[출력 형식 — JSON 배열만, 각 원소는 아래 4개 최상위 키만 포함]
[
  {
    "id": "<원본id>-aug1",
    "input": { "criteria": [...원본과 동일...], "pep_excerpt": "...", "rpt_excerpt": "..." },
    "output": { "label": "<원본과 동일>", "eval": ["완전성: 충족 — ...", "정확성: 불가 — ..."] },
    "_audit": {
      "정확성_판정": "<이 변형의 정확성 판정, criteria에 정확성이 없으면 \\"해당없음\\">",
      "정확성_불가유형": "<정확성이 불가면 1|2|3 중 하나, 충족이거나 criteria에 없으면 \\"해당없음\\">",
      "검증가능성_판정": "<이 변형의 검증가능성 판정, criteria에 없으면 \\"해당없음\\">",
      "검증가능성_블랙리스트포함": "<블랙리스트 표현이 사유에 하나라도 있으면 true, 없으면 false, criteria에 없으면 \\"해당없음\\">"
    }
  }
]
_audit은 채점용 내부 필드다. 반드시 위 4개 키만 채우고, 값은 실제로 네가 생성한 output.eval 내용과 정확히 일치해야 한다.

[최종 점검]
- output.eval의 각 특성 판정이 원본과 한 글자도 다르지 않게 동일한가
- 정확성이 불가면 원본과 같은 유형(1/2/3)을 유지했는가, "기준값 없음"으로 임의 전환하지 않았는가
- 검증가능성에 블랙리스트가 있는데 충족으로 격상시키지 않았는가
- 사유의 모든 수치·숫자쌍이 발췌문에 실제로 있는가
- _audit이 output.eval 내용과 정확히 일치하는가
- JSON 배열 외 텍스트를 출력하지 않았는가

[입력 골든 시드]
{여기에 골든 1건}

위 시드와 동일한 판정을 유지하는 변형 {N}건을 생성하라.
"""


In [ ]:
# ============================================================
# 증강 호출 + 파서
# ============================================================
def parse_json(t):
    t = re.sub(r"^```(json)?|```$", "", t.strip(), flags=re.M).strip()
    return json.loads(t)

def augment(rec, max_retry=2):
    prompt = RPT_AUG_PROMPT.replace("{N}", str(N)) \
             .replace("{여기에 골든 1건}", json.dumps(rec, ensure_ascii=False))
    last_err = None
    for attempt in range(max_retry + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL, temperature=TEMP,
                messages=[{"role":"user","content":prompt}],
            )
            return parse_json(resp.choices[0].message.content)
        except Exception as e:
            last_err = e
            time.sleep(1.5 * (attempt + 1))
    raise last_err


시드 개수: 30


In [ ]:
# ============================================================
# QC 검증
# ============================================================
VALID = {"충족","불가","검토"}
BLACKLIST = ["적절히","최선을 다해","원활히 협의하여","필요한 범위에서","우수한 수준으로","성실히","무리 없이","신속히"]

def parse_eval_map(ev_list):
    d = {}
    for line in ev_list:
        if ":" not in line: continue
        feat = line.split(":")[0].strip()
        rest = line.split(":", 1)[1]
        d[feat] = rest.split("—")[0].strip() if "—" in rest else rest.strip()
    return d

def get_reason(line):
    return line.split("—", 1)[1].strip() if "—" in line else ""

NUM_PATTERN = re.compile(r"\d+\s*(?:일|개월|%|퍼센트|년|월|점|배|명)")
RATIO_PATTERN = re.compile(r"(?<!\.)\d+\s*/\s*\d+")  # 날짜표기(2025.02.14/06.20)는 제외, 순수 분수형만 잡음

def extract_numbers(text):
    text = text or ""
    return set(re.findall(NUM_PATTERN, text)) | set(re.findall(RATIO_PATTERN, text))

def qc(rec, orig):
    if not {"seed_id","id","input","output"}.issubset(rec.keys()): return False, "필드누락"
    inp, out = rec["input"], rec["output"]
    if not {"criteria","pep_excerpt","rpt_excerpt"}.issubset(inp.keys()): return False, "input누락"
    if out.get("label") not in VALID: return False, "label이상"
    if out["label"] != orig["output"]["label"]: return False, "label변경됨"
    if set(inp["criteria"]) != set(orig["input"]["criteria"]): return False, "criteria변경됨"
    ev = out.get("eval")
    if not isinstance(ev, list) or not ev: return False, "eval형식"
    crit = set(inp["criteria"])
    for line in ev:
        feat = line.split(":")[0].strip()
        if feat not in crit: return False, f"criteria밖특성:{feat}"
        if feat == "정확성" and "검토" in line.split("—")[0]: return False, "정확성검토금지"

    # 특성별 판정이 원본과 완전히 동일한지
    orig_map = parse_eval_map(orig["output"]["eval"])
    var_map  = parse_eval_map(ev)
    if orig_map != var_map:
        return False, f"특성별판정불일치:{orig_map}_vs_{var_map}"

    # 근거없는 수치/숫자쌍 삽입 검증
    excerpt_text = inp.get("pep_excerpt","") + inp.get("rpt_excerpt","")
    excerpt_nums = extract_numbers(excerpt_text)
    for line in ev:
        ungrounded = extract_numbers(get_reason(line)) - excerpt_nums
        if ungrounded:
            return False, f"근거없는수치:{ungrounded}"

    # _audit: 정확성 관련만 검증 (VER은 orig_map 비교로 이미 충분히 보장되므로 제외)
    audit = rec.get("_audit", {})
    if audit:
        acc_line = next((l for l in ev if l.split(":")[0].strip() == "정확성"), None)
        if acc_line:
            acc_verdict = acc_line.split(":")[1].split("—")[0].strip()
            if audit.get("정확성_판정") not in ("해당없음", None) and audit.get("정확성_판정") != acc_verdict:
                return False, f"audit불일치:정확성_판정={audit.get('정확성_판정')}_vs_실제={acc_verdict}"
            if acc_verdict == "불가":
                subtype = audit.get("정확성_불가유형")
                if subtype not in ("1", "2", "3"):
                    return False, f"audit불일치:정확성_불가유형이상함({subtype})"

    return True, ""


In [ ]:
# ============================================================
# 메인 증강 루프
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/RPT_augment'
os.makedirs(SAVE_DIR, exist_ok=True)

random.seed(42)
aug, dropped = [], []

used_ids = set()
train_path = f'{SAVE_DIR}/RPT_train.json'
if os.path.exists(train_path):
    try:
        prev = json.load(open(train_path, encoding='utf-8'))
        used_ids |= {r.get('seed_id') for r in prev if r.get('seed_id')}
        print(f"기존 RPT_train.json에서 seed_id {len(used_ids)}건 로드 (충돌 방지용)")
    except Exception as e:
        print("기존 RPT_train.json 로드 실패(무시하고 진행):", e)

for rec in tqdm(seed_train):
    try:
        variants = augment(rec)
    except Exception as e:
        dropped.append({"seed_id": rec["seed_id"], "err": str(e)}); continue

    for i, v in enumerate(variants):
        n = i + 1
        candidate = f"{rec['seed_id']}-aug{n}"
        while candidate in used_ids:
            n += 1
            candidate = f"{rec['seed_id']}-aug{n}"

        v["seed_id"] = candidate
        used_ids.add(candidate)
        v["id"] = rec["id"]

        ok, why = qc(v, rec)
        if not ok:
            dropped.append({"seed_id": v.get("seed_id"), "reason": why}); continue
        v.pop("_audit", None)   # 최종 학습 데이터엔 audit 필드 남기지 않음
        aug.append(v)

train_full = seed_train + aug

seen = set()
deduped = []
dup_removed = 0
for r in train_full:
    key = r.get('seed_id', r.get('id'))
    if key in seen:
        dup_removed += 1
        continue
    seen.add(key)
    deduped.append(r)
train_full = deduped

json.dump(train_full, open(f'{SAVE_DIR}/RPT_train.json', 'w'), ensure_ascii=False, indent=2)
json.dump(dropped, open(f'{SAVE_DIR}/RPT_train_dropped.json', 'w'), ensure_ascii=False, indent=2)

print(f"train시드 {len(seed_train)} + 증강 {len(aug)} = {len(train_full)} / 탈락 {len(dropped)} / 최종중복제거 {dup_removed}")
print("label 분포:", dict(Counter(r['output']['label'] for r in train_full)))
if dropped:
    reason_counts = Counter(d.get('reason', d.get('err','?')).split(':')[0] for d in dropped)
    print("탈락 사유 분포:", dict(reason_counts))
    print("탈락 샘플:", dropped[:5])
print(f"저장 완료: {SAVE_DIR}/RPT_train.json, {SAVE_DIR}/RPT_train_dropped.json")